# 12c Why is brain retrieval weak? Hubness in the one-session benchmark

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 10a, 10b and the four 12a notebooks.

Development analysis, not a prespecified test. 12b's DINOv2 systems on the same 30 sets of 300: how picks concentrate on a few measured patterns, and two
corrections: each measured pattern standardised over predictions for 1,000 reference images outside the test set,
and, as a diagnosis only, over the 300 candidates.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '12c-hubness', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"targets": ["subj01", "subj02", "subj05", "subj07"], "K": 200, "folds": 5, "seed": 13, "ridge_seed": 13, "n_subsets": 30, "subset_size": 300, "subset_seed": 401, "basis_images": 3000, "published": {"MindEye2 (1 session)": [79.0, 57.4], "MindTuner (1 session)": [83.1, 76.0], "MindEye2 (full data)": [98.8, 98.3]}, "ref_images": 1000, "ref_seed": 407}')
SUBJECTS = ['subj01', 'subj02', 'subj03', 'subj04', 'subj05', 'subj06', 'subj07', 'subj08']

import numpy as np
import pandas as pd
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C

IN = Path("/kaggle/input")
conv = next(p.parent for p in IN.rglob("shared_train_nsd_ids.npy"))
common, cols = {}, {}
for h in ("lh", "rh"):
    masks = [np.load(next((conv / f"{s}_roi").rglob(f"{h}.all-vertices_fsaverage_space.npy"))) for s in SUBJECTS]
    common[h], cl = nc.common_vertices(masks)
    cols[h] = dict(zip(SUBJECTS, cl))
V = sum(len(common[h]) for h in common)
NEW = {}
for s in FIXED["targets"]:
    NEW[s] = dict(np.load(next(IN.rglob(f"12a_{s}.npz"))))
    if not (np.array_equal(NEW[s]["common_lh"], common["lh"]) and np.array_equal(NEW[s]["common_rh"], common["rh"])):
        raise ValueError(f"{s}: 12a used other vertices")
shared_ids = NEW[FIXED["targets"][0]]["shared_ids"]
ids, Y, pool_rows = {}, {}, {}
for s in SUBJECTS:
    ids[s] = np.load(conv / f"{s}_train_nsd_ids.npy")
    pool_rows[s] = np.where(~np.isin(ids[s], shared_ids))[0]  # every shared image out of the donors' training
    Y[s] = np.concatenate([np.load(conv / f"{s}_{h}_fmri.npy", mmap_mode="r")[pool_rows[s]][:, cols[h][s]] for h in ("lh", "rh")], 1)
FEATS = {}
for name in ("alexnet", "dinov2"):
    f = np.load(next(IN.rglob(f"10b_{name}_kpca1000.npz")))
    FEATS[name] = (f["z"].astype(np.float32), {int(i): j for j, i in enumerate(f["nsd_ids"])})
feats = lambda name, nsd_ids: FEATS[name][0][[FEATS[name][1][int(i)] for i in nsd_ids]].astype(np.float64)
for s in FIXED["targets"]:
    missing = [int(i) for i in np.r_[NEW[s]["train_ids"], NEW[s]["test_ids"]] if int(i) not in FEATS["alexnet"][1]]
    if missing:
        raise ValueError(f"{s}: {len(missing)} images without features")
print(V, "vertices | donor pools", {s: len(pool_rows[s]) for s in SUBJECTS}, "| new people",
      {s: (len(NEW[s]["train_ids"]), len(NEW[s]["test_ids"])) for s in FIXED["targets"]}, "|", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

In [ ]:
def ols_coef(x, y):
    xa = np.concatenate([np.ones((len(x), 1)), x], 1)
    return np.linalg.solve(xa.T @ xa, xa.T @ np.asarray(y, np.float64)).astype(np.float32)
lin_pred = lambda beta, x: np.concatenate([np.ones((len(x), 1)), x], 1) @ beta.astype(np.float64)
BETA = {}
for f in ("alexnet", "dinov2"):
    for s in SUBJECTS:
        BETA[(f, s)] = ols_coef(feats(f, ids[s][pool_rows[s]]), Y[s])
    print(f, "encoders fitted |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
del Y

In [ ]:
from scipy import stats
SUBSETS = [np.sort(np.random.default_rng([FIXED["subset_seed"], j]).permutation(n)[:FIXED["subset_size"]])
           for j, n in enumerate([len(NEW[FIXED["targets"][0]]["test_ids"])] * FIXED["n_subsets"])]  # as 12b

def zrows(a):
    a = np.asarray(a, np.float64)
    return (a - a.mean(1, keepdims=True)) / (a.std(1, keepdims=True) + 1e-30)

def mean_offdiag(z):
    # mean correlation between distinct rows of a row-z-scored matrix: (|sum of rows|^2 - n V) / (n (n - 1) V)
    n, v = z.shape
    return float(((z.sum(0) ** 2).sum() - n * v) / (n * (n - 1) * v))

def oof(fit_predict, k, seed):
    split = np.array_split(np.random.default_rng(seed).permutation(k), FIXED["folds"])
    out = None
    for held in split:
        pred = fit_predict(np.setdiff1d(np.arange(k), held), held)
        if out is None:
            out = np.empty((k, pred.shape[1]))
        out[held] = pred
    return out

pool_all = np.unique(np.concatenate([ids[s][pool_rows[s]] for s in SUBJECTS]))
basis_ids = np.random.default_rng(FIXED["seed"]).choice(pool_all, FIXED["basis_images"], replace=False)  # as 12b
ref_ids = np.random.default_rng(FIXED["ref_seed"]).choice(pool_all, FIXED["ref_images"], replace=False)  # images only, no shared image
rows = []
for s in FIXED["targets"]:
    t = time.monotonic()
    donors = [d for d in SUBJECTS if d != s]
    tr_ids, te_ids = NEW[s]["train_ids"], NEW[s]["test_ids"]
    y_k = NEW[s]["train_y"].astype(np.float64)
    m = zrows(NEW[s]["test_mean"])
    zk = zrows(y_k)
    k, d, V_ = len(tr_ids), FIXED["seed"], m.shape[1]
    def record(system, pred, pred_ref):
        p, pr = zrows(pred), zrows(pred_ref)
        # reference statistics: each measured pattern over the reference images' predictions (no test response),
        # each prediction over the person's training trials (their labelled data)
        r_mr = m @ pr.T / V_
        mu_m, sd_m = r_mr.mean(1), r_mr.std(1) + 1e-12
        r_pk = p @ zk.T / V_
        mu_p, sd_p = r_pk.mean(1), r_pk.std(1) + 1e-12
        acc = {key: [] for key in ("image", "brain", "brain_rowcentre", "brain_rowz", "brain_ref", "image_ref", "hub_top3", "hub_never",
                                   "hub_skew", "hub_vs_mean_sim", "img_hub_top3", "pred_offdiag", "meas_offdiag")}
        for S in SUBSETS:
            r = m[S] @ p[S].T / V_  # rows: measured patterns, columns: predictions
            eye = np.arange(len(S))
            acc["image"].append((r.argmax(1) == eye).mean() * 100)
            acc["brain"].append((r.argmax(0) == eye).mean() * 100)
            rc = r - r.mean(1, keepdims=True)
            acc["brain_rowcentre"].append((rc.argmax(0) == eye).mean() * 100)
            acc["brain_rowz"].append(((rc / (r.std(1, keepdims=True) + 1e-12)).argmax(0) == eye).mean() * 100)
            acc["brain_ref"].append((((r - mu_m[S][:, None]) / sd_m[S][:, None]).argmax(0) == eye).mean() * 100)
            acc["image_ref"].append((((r - mu_p[S][None]) / sd_p[S][None]).argmax(1) == eye).mean() * 100)
            counts = np.bincount(r.argmax(0), minlength=len(S))
            acc["hub_top3"].append(np.sort(counts)[-3:].sum() / len(S) * 100)
            acc["hub_never"].append((counts == 0).mean() * 100)
            acc["hub_skew"].append(float(stats.skew(counts)))
            acc["hub_vs_mean_sim"].append(float(stats.spearmanr(counts, r.mean(1)).correlation))
            acc["img_hub_top3"].append(np.sort(np.bincount(r.argmax(1), minlength=len(S)))[-3:].sum() / len(S) * 100)
            acc["pred_offdiag"].append(mean_offdiag(p[S]))
            acc["meas_offdiag"].append(mean_offdiag(m[S]))
        out = {key: float(np.mean(v)) for key, v in acc.items()}
        rows.append({"subject": s, "system": system, **out})
        print(f"  {s} {system:10s} image {out['image']:.1f} (ref {out['image_ref']:.1f}) | brain {out['brain']:.1f} -> ref {out['brain_ref']:.1f},"
              f" row-centre {out['brain_rowcentre']:.1f}, row-z {out['brain_rowz']:.1f} | top-3 hubs take {out['hub_top3']:.1f}%,"
              f" never picked {out['hub_never']:.1f}% | offdiag pred {out['pred_offdiag']:.3f} meas {out['meas_offdiag']:.3f}", flush=True)
    xs = {f: (feats(f, tr_ids), feats(f, te_ids), feats(f, ref_ids)) for f in ("alexnet", "dinov2")}
    def avg_of(x):
        acc = np.zeros((len(x), V))
        for dn in donors:
            acc += lin_pred(BETA[("dinov2", dn)], x)
        return acc / len(donors)
    x_k, x_t, x_r = xs["dinov2"]
    a_k, a_t, a_r = avg_of(x_k), avg_of(x_t), avg_of(x_r)
    mean, B = lm.prediction_basis(avg_of(feats("dinov2", basis_ids)), FIXED["K"])
    record("borrowing", a_t - a_k.mean(0), a_r - a_k.mean(0))
    model = lm.fit_lowrank_map(a_k, y_k, mean, B, seed=d)
    m_t, m_r = lm.apply_lowrank_map(model, a_t, mean, B), lm.apply_lowrank_map(model, a_r, mean, B)
    record("map", m_t, m_r)
    m_oof = oof(lambda tr, he: lm.apply_lowrank_map(lm.fit_lowrank_map(a_k[tr], y_k[tr], mean, B, seed=d), a_k[he], mean, B), k, d)
    own = {}
    for f in ("alexnet", "dinov2"):
        fx_k, fx_t, fx_r = xs[f]
        o = RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(fx_k, y_k)
        own[f] = (oof(lambda tr, he: o.refit(fx_k[tr], y_k[tr]).predict(fx_k[he]), k, d), o.predict(fx_t), o.predict(fx_r))
    record("own_ridge", own["dinov2"][1], own["dinov2"][2])
    w = nc.stack_fit([m_oof, own["alexnet"][0]], y_k)
    record("mixed", nc.stack_apply(w, [m_oof, own["alexnet"][0]], [m_t, own["alexnet"][1]], y_k.mean(0)),
           nc.stack_apply(w, [m_oof, own["alexnet"][0]], [m_r, own["alexnet"][2]], y_k.mean(0)))
    record("own_full", lin_pred(BETA[("dinov2", s)], x_t), lin_pred(BETA[("dinov2", s)], x_r))
    del a_k, a_t, a_r, m_oof, own
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "12c_scores.csv", index=False)
    print(s, f"| {time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

In [ ]:
frame = pd.DataFrame(rows)
res = {"post_hoc": "development: why brain retrieval is weak in 12b; ref-norm is protocol-clean, row-norm uses the subset",
       "systems": {}}
for system, g in frame.groupby("system", sort=False):
    g = g.set_index("subject").reindex(FIXED["targets"])
    res["systems"][system] = {c: float(g[c].mean()) for c in frame.columns if c not in ("subject", "system")}
    for c in ("brain_ref", "brain_rowz", "image_ref"):
        base = "image" if c.startswith("image") else "brain"
        x = g[c] - g[base]
        res["systems"][system][f"{c}_minus_{base}"] = {"mean": float(x.mean()), "positive": int((x > 0).sum()), "n": int(x.notna().sum())}
res["check_against_12b"] = {"mixed_image": res["systems"]["mixed"]["image"], "mixed_brain": res["systems"]["mixed"]["brain"],
                            "12b": [64.5, 34.2]}
(C.METRICS_DIR / "12c_summary.json").write_text(json.dumps(res, indent=2))
print("12C_SUMMARY_JSON " + json.dumps(res))